# LIBRARY

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [4]:
import pandas as pd

def analyze_feature(df, colonna, quantili=[0.1,0.25,0.50, 0.75,0.9,0.99]):
    serie = df[colonna] 
    
    # CAL values equal to 0
    num_zero = (serie == 0).sum()
    presenza_zero = num_zero > 0
    
    # CAL negative values
    num_negativi = (serie < 0).sum()
    presenza_negativi = num_negativi > 0
    
    # CAL range (min and max)
    valore_min = serie.min()
    valore_max = serie.max()
    range_valori = valore_max - valore_min

     # CAL quantiles
    valori_quantili = serie.quantile(quantili)
    
    # Stampa risultati
    print(f"Analyzed column: {colonna}")
    print(f"Presence of values equal to 0: {presenza_zero}")
    print(f"Number of values equal to 0: {num_zero}")
    print(f"Presence of negative values: {presenza_negativi}")
    print(f"Number of negative values: {num_negativi}")    
    print(f"Minimum value: {valore_min}")
    print(f"Maximum value: {valore_max}")
    print(f"Range (max - min): {range_valori}")

    print(f"\nQuantiles:")
    for q, v in valori_quantili.items():
        print(f"  Q{int(q*100)} ({q}): {v}")

# FEATURE SELECTION

## read df

In [5]:
file_path = "../1.DATASET/cmi_internet.csv"
df_clean=pd.read_csv(file_path)

In [6]:
df_clean=df_clean[['id',
    'Basic_Demos-Age', 
    'Basic_Demos-Sex',
    'CGAS-CGAS_Score', 
    'Physical-BMI',
    'Physical-Height', 
    'Physical-Weight',
    'Physical-Waist_Circumference',
    'Physical-Diastolic_BP',
    'Physical-HeartRate', 
    'Physical-Systolic_BP',
    'Fitness_Endurance-Max_Stage',
    'FGC-FGC_CU', 
    'FGC-FGC_GSND',
    'FGC-FGC_GSD', 
    'FGC-FGC_PU',
    'FGC-FGC_SRL', 
    'FGC-FGC_SRR',
    'FGC-FGC_TL', 
    'BIA-BIA_Activity_Level_num',
    'BIA-BIA_BMI',
    'BIA-BIA_BMC',
    'BIA-BIA_DEE',
    'BIA-BIA_FFMI', 
    'BIA-BIA_FMI', 
    'BIA-BIA_Fat',
    'BIA-BIA_Frame_num',
    'BIA-BIA_SMM',
    'BIA-BIA_TBW', 
    'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02',
    'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06',
    'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10',
    'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14',
    'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18',
    'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'PCIAT-PCIAT_Total', 
    'SDS-SDS_Total_T', 
    'PreInt_EduHx-computerinternet_hoursday', 'sii']]

## select features

### PCIAT-PCIAT_Total_CAL

In [7]:
### ASK TO MANY NULL

In [8]:
cols_to_sum = ['PCIAT-PCIAT_01', 'PCIAT-PCIAT_02',
       'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06',
       'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10',
       'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14',
       'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18',
       'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20']
df_clean['PCIAT-PCIAT_Total_CAL']=df_clean[cols_to_sum].sum(axis=1)

df_clean[['PCIAT-PCIAT_Total','PCIAT-PCIAT_Total_CAL']][(round(df_clean['PCIAT-PCIAT_Total_CAL'],0)!=round(df_clean['PCIAT-PCIAT_Total'],0))
                                                        & (df_clean['PCIAT-PCIAT_Total_CAL']!=0)]

,PCIAT-PCIAT_Total,PCIAT-PCIAT_Total_CAL
24,30.0,31.0
141,26.0,27.0
255,81.0,82.0
270,48.0,49.0
425,5.0,7.0
944,53.0,54.0
1120,56.0,59.0
1247,40.0,41.0
1387,64.0,65.0
1472,74.0,75.0


In [9]:
analyze_feature(df_clean, 'PCIAT-PCIAT_Total_CAL')

Analyzed column: PCIAT-PCIAT_Total_CAL
Presence of values equal to 0: True
Number of values equal to 0: 6054
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 93.0
Range (max - min): 93.0

Quantiles:
  Q10 (0.1): 0.0
  Q25 (0.25): 0.0
  Q50 (0.5): 0.0
  Q75 (0.75): 9.0
  Q90 (0.9): 36.0
  Q99 (0.99): 71.0


### Mean Arterial Pressure (MAP)

In [10]:
df_clean['Physical-MAP_CAL'] = df_clean.apply(lambda x:
        x['Physical-Diastolic_BP']+((x['Physical-Systolic_BP']-x['Physical-Diastolic_BP'])/3)
        if x['Physical-Diastolic_BP']<=x['Physical-Systolic_BP'] 
        else x['Physical-Systolic_BP']+((x['Physical-Diastolic_BP']-x['Physical-Systolic_BP'])/3)
        , axis=1)

analyze_feature(df_clean, 'Physical-MAP_CAL')

Analyzed column: Physical-MAP_CAL
Presence of values equal to 0: True
Number of values equal to 0: 1
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 164.33333333333334
Range (max - min): 164.33333333333334

Quantiles:
  Q10 (0.1): 73.66666666666667
  Q25 (0.25): 78.5
  Q50 (0.5): 83.16666666666667
  Q75 (0.75): 88.83333333333333
  Q90 (0.9): 97.0
  Q99 (0.99): 122.33333333333333


### FGC-FGC_CU and FGC-FGC_PU

In [11]:
df_clean['FGC-FGC_CORE_CAL']=df_clean['FGC-FGC_CU']+df_clean['FGC-FGC_PU']
analyze_feature(df_clean, 'FGC-FGC_CORE_CAL')

Analyzed column: FGC-FGC_CORE_CAL
Presence of values equal to 0: True
Number of values equal to 0: 529
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 150.0
Range (max - min): 150.0

Quantiles:
  Q10 (0.1): 1.0
  Q25 (0.25): 5.0
  Q50 (0.5): 12.0
  Q75 (0.75): 20.0
  Q90 (0.9): 32.0
  Q99 (0.99): 62.0


### FGC-FGC_GSND and FGC-FGC_GSD

In [12]:
df_clean['FGC-FGC_GS_CAL']=(df_clean['FGC-FGC_GSND']+df_clean['FGC-FGC_GSD'])/2
analyze_feature(df_clean, 'FGC-FGC_GS_CAL')

Analyzed column: FGC-FGC_GS_CAL
Presence of values equal to 0: True
Number of values equal to 0: 3
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 115.1
Range (max - min): 115.1

Quantiles:
  Q10 (0.1): 13.975
  Q25 (0.25): 17.518749999999997
  Q50 (0.5): 18.875
  Q75 (0.75): 22.1
  Q90 (0.9): 29.475
  Q99 (0.99): 47.0065


### FGC-FGC_SRL and FGC-FGC_SRR

In [13]:
df_clean['FGC-FGC_SR_CAL']=(df_clean['FGC-FGC_SRR']+df_clean['FGC-FGC_SRL'])/2
analyze_feature(df_clean, 'FGC-FGC_SR_CAL')

Analyzed column: FGC-FGC_SR_CAL
Presence of values equal to 0: True
Number of values equal to 0: 87
Presence of negative values: False
Number of negative values: 0
Minimum value: 0.0
Maximum value: 21.0
Range (max - min): 21.0

Quantiles:
  Q10 (0.1): 5.0
  Q25 (0.25): 7.0
  Q50 (0.5): 8.5
  Q75 (0.75): 9.5
  Q90 (0.9): 11.25
  Q99 (0.99): 15.0


In [14]:
df_clean.columns

Index(['id', 'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-BMI', 'Physical-Height', 'Physical-Weight',
       'Physical-Waist_Circumference', 'Physical-Diastolic_BP',
       'Physical-HeartRate', 'Physical-Systolic_BP',
       'Fitness_Endurance-Max_Stage', 'FGC-FGC_CU', 'FGC-FGC_GSND',
       'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMI', 'BIA-BIA_BMC',
       'BIA-BIA_DEE', 'BIA-BIA_FFMI', 'BIA-BIA_FMI', 'BIA-BIA_Fat',
       'BIA-BIA_Frame_num', 'BIA-BIA_SMM', 'BIA-BIA_TBW', 'PCIAT-PCIAT_01',
       'PCIAT-PCIAT_02', 'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05',
       'PCIAT-PCIAT_06', 'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09',
       'PCIAT-PCIAT_10', 'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13',
       'PCIAT-PCIAT_14', 'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17',
       'PCIAT-PCIAT_18', 'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20',
       'PCIAT-PC

## DROP ATTRIBUITES

In [15]:
df_clean=df_clean.drop(columns = 
  [
      'Physical-Diastolic_BP','Physical-Systolic_BP', 'FGC-FGC_CU', 'FGC-FGC_GSND',
      'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR',
      'PCIAT-PCIAT_01', 'PCIAT-PCIAT_02',
      'PCIAT-PCIAT_03', 'PCIAT-PCIAT_04', 'PCIAT-PCIAT_05', 'PCIAT-PCIAT_06',
      'PCIAT-PCIAT_07', 'PCIAT-PCIAT_08', 'PCIAT-PCIAT_09', 'PCIAT-PCIAT_10',
      'PCIAT-PCIAT_11', 'PCIAT-PCIAT_12', 'PCIAT-PCIAT_13', 'PCIAT-PCIAT_14',
      'PCIAT-PCIAT_15', 'PCIAT-PCIAT_16', 'PCIAT-PCIAT_17', 'PCIAT-PCIAT_18',
      'PCIAT-PCIAT_19', 'PCIAT-PCIAT_20', 'Physical-Waist_Circumference'
  ])

In [16]:
df_clean.shape

(8460, 29)

In [17]:
df_clean.describe()

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_TBW,PCIAT-PCIAT_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
count,8460.000000,8460.000000,8460.000000,7034.000000,7591.000000,7595.000000,7641.000000,7539.000000,5480.000000,6945.000000,...,6636.000000,2714.000000,4903.000000,7849.000000,8460.000000,8460.000000,6877.000000,5916.000000,4188.000000,5945.000000
mean,4229.500000,10.240189,0.402364,67.021041,19.532651,57.189069,84.428737,81.894150,4.918978,8.878107,...,51.694005,27.855195,58.072813,0.981144,0.443853,8.939598,84.606006,14.778736,20.624355,8.322321
std,2442.335972,3.574680,0.490404,35.284140,4.683925,7.374577,40.218441,11.380533,1.174191,2.558232,...,132.840059,20.318784,12.563287,1.046645,0.731014,17.370330,10.761730,13.785133,7.319546,2.606244
min,0.000000,5.000000,0.000000,25.000000,0.000000,33.000000,0.000000,27.000000,0.000000,0.000000,...,20.589200,0.000000,38.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2114.750000,7.000000,0.000000,60.500000,16.541742,51.500000,55.200000,75.500000,4.500000,8.000000,...,34.802217,12.000000,50.000000,0.000000,0.000000,0.000000,78.500000,5.000000,17.518750,7.000000
50%,4229.500000,10.000000,0.000000,65.000000,17.937682,55.700000,75.000000,81.500000,5.000000,9.000000,...,44.987000,26.000000,55.000000,1.000000,0.000000,0.000000,83.166667,12.000000,18.875000,8.500000
75%,6344.250000,12.000000,1.000000,70.500000,21.469546,63.000000,107.000000,87.000000,5.000000,10.000000,...,53.422779,41.000000,62.500000,2.000000,1.000000,9.000000,88.833333,20.000000,22.100000,9.500000
max,8459.000000,22.000000,1.000000,999.000000,59.132048,78.500000,315.000000,138.000000,28.000000,22.000000,...,5690.910000,93.000000,100.000000,3.000000,3.000000,93.000000,164.333333,150.000000,115.100000,21.000000


In [18]:
df_clean.columns

Index(['id', 'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
       'Physical-BMI', 'Physical-Height', 'Physical-Weight',
       'Physical-HeartRate', 'Fitness_Endurance-Max_Stage', 'FGC-FGC_TL',
       'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMI', 'BIA-BIA_BMC',
       'BIA-BIA_DEE', 'BIA-BIA_FFMI', 'BIA-BIA_FMI', 'BIA-BIA_Fat',
       'BIA-BIA_Frame_num', 'BIA-BIA_SMM', 'BIA-BIA_TBW', 'PCIAT-PCIAT_Total',
       'SDS-SDS_Total_T', 'PreInt_EduHx-computerinternet_hoursday', 'sii',
       'PCIAT-PCIAT_Total_CAL', 'Physical-MAP_CAL', 'FGC-FGC_CORE_CAL',
       'FGC-FGC_GS_CAL', 'FGC-FGC_SR_CAL'],
      dtype='object')

# CONTROL DATASET 

In [19]:
file_path = "../2.DATA EXPLORATION/control_id.csv"
df_id=pd.read_csv(file_path)

In [20]:
df_train=df_clean.copy()
df_train

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_TBW,PCIAT-PCIAT_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
0,0,5,0,51.0,16.877316,46.00,50.8,NaN,5.0,6.0,...,32.690900,55.0,NaN,3.0,2.0,55.0,NaN,0.0,NaN,6.5
1,1,9,0,NaN,14.035590,48.00,46.0,70.0,NaN,3.0,...,27.055200,0.0,64.0,0.0,0.0,0.0,90.666667,8.0,NaN,11.0
2,2,10,1,71.0,16.648696,56.50,75.6,94.0,5.0,5.0,...,44.987000,28.0,54.0,2.0,0.0,28.0,82.333333,27.0,12.450,10.0
3,3,9,0,71.0,18.292347,56.00,81.6,97.0,6.0,7.0,...,45.996600,44.0,45.0,0.0,1.0,44.0,79.000000,23.0,NaN,7.0
4,4,18,1,65.0,17.937682,NaN,77.0,NaN,NaN,10.0,...,NaN,NaN,NaN,1.0,0.0,0.0,NaN,12.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8455,8455,7,1,NaN,16.130585,46.07,49.0,82.5,2.5,8.0,...,31.583004,NaN,55.0,0.0,0.0,0.0,NaN,6.5,18.075,7.0
8456,8456,10,1,69.5,NaN,56.13,47.8,80.5,5.0,8.0,...,36.475662,NaN,NaN,0.0,1.0,0.0,79.833333,8.5,13.575,5.0
8457,8457,10,1,70.0,40.937571,49.56,47.2,83.5,7.0,NaN,...,35.106855,NaN,NaN,2.0,0.0,0.0,82.166667,21.5,NaN,9.5
8458,8458,15,1,55.5,NaN,63.79,99.5,87.5,NaN,9.0,...,57.530686,NaN,NaN,1.0,2.0,0.0,NaN,10.0,NaN,10.5


In [21]:
df_train_control = pd.merge(df_train, df_id, on=['id'], how='inner')
df_train_control

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_TBW,PCIAT-PCIAT_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
0,1,9,0,NaN,14.035590,48.00,46.0,70.0,NaN,3.0,...,27.055200,0.0,64.0,0.0,0.0,0.0,90.666667,8.0,NaN,11.0
1,3,9,0,71.0,18.292347,56.00,81.6,97.0,6.0,7.0,...,45.996600,44.0,45.0,0.0,1.0,44.0,79.000000,23.0,NaN,7.0
2,5,13,1,50.0,22.279952,59.50,112.2,73.0,5.0,8.0,...,63.126500,34.0,56.0,0.0,1.0,34.0,74.000000,18.0,17.200,10.5
3,6,10,0,NaN,19.660760,55.00,84.6,83.0,NaN,11.0,...,47.221100,20.0,40.0,3.0,0.0,20.0,136.333333,11.0,NaN,11.0
4,7,10,1,65.0,16.861286,59.25,84.2,90.0,5.0,4.0,...,50.476700,NaN,55.0,2.0,0.0,0.0,86.000000,0.0,11.850,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2131,8371,6,0,65.5,15.496182,46.84,64.7,83.5,6.0,NaN,...,32.771756,NaN,NaN,1.0,0.0,0.0,76.000000,0.0,15.025,7.5
2132,8406,10,1,61.5,17.057230,58.32,75.1,82.5,4.0,7.0,...,43.230003,NaN,46.0,0.0,0.0,0.0,111.666667,NaN,NaN,3.0
2133,8426,7,0,77.0,18.020412,64.24,72.4,87.5,NaN,9.0,...,40.690369,NaN,NaN,0.0,0.0,0.0,80.500000,NaN,NaN,8.5
2134,8440,10,1,59.5,17.322879,55.10,72.0,75.5,4.5,7.0,...,41.114615,NaN,NaN,2.0,0.0,0.0,86.166667,11.5,NaN,8.5


In [23]:
df_train_control.to_csv('CMI_V2_CONTROL.csv', index=False)

# FULL DATASET 

In [24]:
df_clean.shape

(8460, 29)

In [25]:
df_clean

,id,Basic_Demos-Age,Basic_Demos-Sex,CGAS-CGAS_Score,Physical-BMI,Physical-Height,Physical-Weight,Physical-HeartRate,Fitness_Endurance-Max_Stage,FGC-FGC_TL,...,BIA-BIA_TBW,PCIAT-PCIAT_Total,SDS-SDS_Total_T,PreInt_EduHx-computerinternet_hoursday,sii,PCIAT-PCIAT_Total_CAL,Physical-MAP_CAL,FGC-FGC_CORE_CAL,FGC-FGC_GS_CAL,FGC-FGC_SR_CAL
0,0,5,0,51.0,16.877316,46.00,50.8,NaN,5.0,6.0,...,32.690900,55.0,NaN,3.0,2.0,55.0,NaN,0.0,NaN,6.5
1,1,9,0,NaN,14.035590,48.00,46.0,70.0,NaN,3.0,...,27.055200,0.0,64.0,0.0,0.0,0.0,90.666667,8.0,NaN,11.0
2,2,10,1,71.0,16.648696,56.50,75.6,94.0,5.0,5.0,...,44.987000,28.0,54.0,2.0,0.0,28.0,82.333333,27.0,12.450,10.0
3,3,9,0,71.0,18.292347,56.00,81.6,97.0,6.0,7.0,...,45.996600,44.0,45.0,0.0,1.0,44.0,79.000000,23.0,NaN,7.0
4,4,18,1,65.0,17.937682,NaN,77.0,NaN,NaN,10.0,...,NaN,NaN,NaN,1.0,0.0,0.0,NaN,12.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8455,8455,7,1,NaN,16.130585,46.07,49.0,82.5,2.5,8.0,...,31.583004,NaN,55.0,0.0,0.0,0.0,NaN,6.5,18.075,7.0
8456,8456,10,1,69.5,NaN,56.13,47.8,80.5,5.0,8.0,...,36.475662,NaN,NaN,0.0,1.0,0.0,79.833333,8.5,13.575,5.0
8457,8457,10,1,70.0,40.937571,49.56,47.2,83.5,7.0,NaN,...,35.106855,NaN,NaN,2.0,0.0,0.0,82.166667,21.5,NaN,9.5
8458,8458,15,1,55.5,NaN,63.79,99.5,87.5,NaN,9.0,...,57.530686,NaN,NaN,1.0,2.0,0.0,NaN,10.0,NaN,10.5


In [26]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8460 entries, 0 to 8459
Data columns (total 29 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   id                                      8460 non-null   int64  
 1   Basic_Demos-Age                         8460 non-null   int64  
 2   Basic_Demos-Sex                         8460 non-null   int64  
 3   CGAS-CGAS_Score                         7034 non-null   float64
 4   Physical-BMI                            7591 non-null   float64
 5   Physical-Height                         7595 non-null   float64
 6   Physical-Weight                         7641 non-null   float64
 7   Physical-HeartRate                      7539 non-null   float64
 8   Fitness_Endurance-Max_Stage             5480 non-null   float64
 9   FGC-FGC_TL                              6945 non-null   float64
 10  BIA-BIA_Activity_Level_num              6636 non-null   floa

In [27]:
df_clean.to_csv('CMI_V2.csv', index=False)